<a href="https://colab.research.google.com/github/juampamuc/notebooks/blob/main/Automating_Medical_Image_Processing_with_3D_Slicer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automating Medical Image Processing with 3D Slicer

## Introduction

If you ever had to visualize medical data, you probably already interacted with [3D Slicer](https://www.slicer.org/). If not, let me introduce this wonderful tool. 3D Slicer is a **free**, open source and multi-platform software, that allows you to display medical data (DICOM, Nifti, ...) in 2D or even in 3D. You can also do image processing such as cropping, skull stripping and many more. You can even use it to segment medical data yourself!

However, certain task remain quite repetitive. For instance, combing through an entire dataset for a challenge may involve repeating the same actions dozens of time. But fear not! About everything can be automated in 3D Slicer with a bit of Python scripting. This notebook is here to show you the basics and give you some ideas.

## Prerequisites

The first step is of course to install 3D Slicer. If it's not already done, you can download it [here](https://download.slicer.org/). Once 3D Slicer is running on your computer, you have two options to start scripting:
- **Option 1:** Use the Python console bundled with 3D Slicer. No further configuration is required, simply click the Python button at the top of your screen (or press Ctrl+3). ![image.png](attachment:image.png) You can then write your scripts in your favorite IDE and copy-paste them later in the Python Interactor at the bottom of your window. The main downside is that the code clocks may not contain any empty lines. So either pack your code or put empty comments in place of empty lines.
- **Option 2:** Link your Jupyter Server to 3D Slicer. Configuring this is quite easy and can be done in a few minutes. Simply follow the short [tutorial](https://www.youtube.com/watch?v=jcRsRw6RC2g) (credits @PerkLab Research). The main advantage is that Jupyter Notebooks provide for a more comfortable development environment, as all the 3D Slicer specific libraries are installed in the bundled Python distribution. You can also build a repository of small helper scripts that you can easily execute in a click.

**Note:** If you choose Option 2, you will also be able to directly run this Notebook. To this end, you should run the next code block to download some sample data for the demo. We will go over the different functionalities  later, don't worry.

In [ ]:
!pip install SampleData

In [ ]:
### Install Required Libraries (Run this first in Google Colab)
!pip install SimpleITK ipywidgets opencv-python-headless matplotlib numpy


In [ ]:
!apt-get update
!apt-get install -y fsl

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,077 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,584 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,751 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-d

In [ ]:
# Step 1: Update package lists and install FSL
!apt-get update -qq
!apt-get install -y fsl

# Step 2: Set up FSL environment variables
import os
os.environ["FSLDIR"] = "/usr/share/fsl/5.0"
os.environ["PATH"] += ":/usr/lib/fsl/5.0"

# Step 3: Source the FSL configuration file
!source $FSLDIR/etc/fslconf/fsl.sh

print("FSL installed successfully.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package fsl is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source

E: Package 'fsl' has no installation candidate
/bin/bash: line 1: /usr/share/fsl/5.0/etc/fslconf/fsl.sh: No such file or directory
FSL installed successfully.


In [ ]:
input_file_path ="/sample_data/chris_PD.nii.gz"

In [ ]:
!bet {input_file_path} /content/skull_stripped.nii.gz -m

/bin/bash: line 1: bet: command not found


In [ ]:
# Check if FSL binaries are accessible
!which bet
!which fslinfo

# Test the 'fslinfo' command
!fslinfo /usr/share/fsl/5.0/data/standard/MNI152_T1_1mm.nii.gz

/bin/bash: line 1: fslinfo: command not found


In [ ]:
!

In [ ]:
# Step 1: Install ANTsPy
!pip install antspyx

import ants
from google.colab import files

# Step 2: Upload your clinical data
# uploaded = files.upload()
input_file_path = "/content/sub-01_ses-01_task-Stroop_acq-cf1PA_run-02_bold.nii"

# Step 3: Load the image
image = ants.image_read(input_file_path)

# Step 4: Perform skull stripping using Atropos (brain extraction)
# Atropos is a segmentation tool that can be used for skull stripping
segmentation = ants.atropos(a=image, mrf=0.1, initialization='kmeans[3]')
brain_mask = segmentation['probabilityimages'][2]  # Extract the brain mask

# Step 5: Apply the brain mask to the original image
skull_stripped = image * brain_mask

# Step 6: Save the results
output_volume_path = "/content/skull_stripped.nii.gz"
output_mask_path = "/content/brain_mask.nii.gz"

ants.image_write(skull_stripped, output_volume_path)
ants.image_write(brain_mask, output_mask_path)

print(f"Skull stripping completed. Output saved to:\n{output_volume_path}\n{output_mask_path}")

# Step 7: Download the results
files.download(output_volume_path)  # Download skull-stripped volume
files.download(output_mask_path)    # Download brain mask

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2


ValueError: File /content/sub-01_ses-01_task-Stroop_acq-cf1PA_run-02_bold.nii does not exist!

In [ ]:
!pip uninstall -y antspyx

In [ ]:
!pip install antspyx

In [ ]:
import ants
print(ants.__version__)

In [ ]:
import ants

# Check if the brain_extraction function exists
if hasattr(ants, "brain_extraction"):
    print("The brain_extraction function is available.")
else:
    print("The brain_extraction function is NOT available.")

In [ ]:
# Step 1: Install ANTsPy
# !pip install antspyx

import ants
from google.colab import files

# Step 2: Upload your clinical data
# uploaded = files.upload()
input_file_path ="/content/852176.nii.gz" # Get the uploaded file name

# Step 3: Load the image
image = ants.image_read(input_file_path)

# Step 4: Perform skull stripping using the BrainExtraction tool
# Use the pre-trained brain extraction model
brain_extraction = ants.brain_extraction(image, "t1")  # "t1" specifies the modality (T1-weighted MRI)

# Extract the brain mask and skull-stripped image
brain_mask = brain_extraction["segmentation"]  # Binary brain mask
skull_stripped = brain_extraction["brain_image"]  # Skull-stripped brain

# Step 5: Save the results
output_volume_path = "/content/skull_stripped.nii.gz"
output_mask_path = "/content/brain_mask.nii.gz"

ants.image_write(skull_stripped, output_volume_path)
ants.image_write(brain_mask, output_mask_path)

print(f"Skull stripping completed. Output saved to:\n{output_volume_path}\n{output_mask_path}")

# Step 6: Download the results
files.download(output_volume_path)  # Download skull-stripped volume
files.download(output_mask_path)    # Download brain mask

## Loading data

### The Manual Way

Since you can use 3D Slicer as you would usually do. You can choose to load your data manually. So, you can still drag and drop DICOM folders into your DICOM database...
![image.png](attachment:image.png)
...or you can drag and drop your Nifti files:
![image2.png](attachment:image2.png)
But one can, already see that having to check "Segmentation" for each segmentation file can be tedious when combing through an entire dataset.

### With Python

Just like before, you can either iterate over your DICOM database and use the DICOM importer:

In [ ]:
from DICOMLib import DICOMUtils

# Get you DICOM database
db = slicer.dicomDatabase
# For each patient
for patient in db.patients():
    # For each study for the patient
    for study in db.studiesForPatient(patient):
        # For each serie in the study
        for serie in db.seriesForStudy(study):
            # Get file list and serie info
            file_list = db.filesForSeries(serie)
            # You can also access any information in the DICOM header (see the official documentation)
            serie_name = db.fileValue(file_list[0], "0008,103E")

            # Load with verification
            DICOMUtils.loadSeriesWithVerification([serie])

            # Or if the verification fails...
            loadablesByPlugin, _ = DICOMUtils.getLoadablesFromFileLists([file_list])
            nodes = DICOMUtils.loadLoadables(loadablesByPlugin)
            print("Nodes:", nodes)


As you can see, loading DICOM files from your database is quite straight forward. And if the files are not yet in your database, you only require the list of file paths to load them. Also note, that only `DICOMUtils.getLoadablesFromFileLists` will return the list of nodes that it loaded. If you use the first method, you need the `getNode(node_name)` method to retrieve the handles to the data nodes by name.

In contrast, handling Nifti files is much easier. 3D Slicer will select the correct laoder based on the file extension and will also return the data node that was loaded:

In [ ]:
# Load the Nifti volume
volume = slicer.util.loadVolume(sample_nifti_path)
print("Volume node:", repr(volume))
# Load the segmentation
segmentation = slicer.util.loadSegmentation(sample_segmentation_path)
print("Segmentation node:", repr(segmentation))

## Nodes and Node Hierarchy

Before, we start doing any image processing, let's talk a bit more about nodes and how they are structured. This will be useful for later, for instance if you have to create new volumes or segmentations to store the results computed by a module.

### The Subject Hierachy

As you have already guessed, the nodes a stored in a tree-shaped structure. You will have noticed that data loaded from DICOM files follow a specific pattern:
- Patient Node: contains all of the patient information
  - Study Nodes: each contains all data measured for a specific study, e.g. the modalities recorded on a certain date
    - Data Nodes: each contains a 2D or 3D image. It can either be a volume for a specific modality (e.g. MR T1w, CT...) or a segmentation of a volume

Now, this ordering does not have to be respected and you can have patient nodes that are children of other patient nodes. This is only important, if you are working with DICOM data. But, if you peeked in the setup code, you have already seen that adding new patients or series is really easy:

In [ ]:
# Get the hierarchy node for the scene
shNode = slicer.vtkMRMLSubjectHierarchyNode.GetSubjectHierarchyNode(slicer.mrmlScene)
# Using the hierachy node, you can create a new patient
# You only need:
#  - the id of the parent node (in this case the root of the tree)
#  - a name for your patient node
# The function will then return the ID of your node
another_patient_id = shNode.CreateSubjectItem(shNode.GetSceneItemID(), "Another Sample Patient")
# Using the hierachy node, you can then create a new serie
# It works exactly like creating a new patient node
another_study_id = shNode.CreateStudyItem(another_patient_id, "Another Sample Study")

### Data Nodes

You can also as easily add new data nodes to the scene. You will only require the class name of the node that you want to add. Here are a few examples, but you can find an exhaustive list in [3D Slicer's C++ Documentation](https://apidocs.slicer.org/master/annotated.html):
- Volume Node: "vtkMRMLScalarVolumeNode"
- Segmentation Node: "vtkMRMLSegmentationNode"
- Label Map Node: "vtkMRMLLabelMapVolumeNode"

In [ ]:
# Get the scene object
scene = slicer.mrmlScene
# Add a new node
# The function will return the node object
new_volume_node = scene.AddNewNodeByClass("vtkMRMLScalarVolumeNode")
print("New Volume Node:", repr(new_volume_node))

You can also just as easily move nodes in the hierarchy, only the ID of the node you want to move, and the ID of its new parent are required:

In [ ]:
# Here, we are moving the sample data node "sample_volume" that we created earlier to be a child of the new study node
volume_id = shNode.GetItemByDataNode(volume)
shNode.SetItemParent(volume_id, another_study_id)

## Image Processing

If you are here doing the image processing through 3D Slicer, then you most likely want to run 3D Slicer modules automatically. And that's exactly what we are going to do! In particular, we will be doing skull stripping using the "SwissSkullStripper" module. So if you want, make sure that this module is installed before running this notebook.
![image.png](attachment:image.png)
Once this module has been installed, we can first quickly take a look at the user-friendly interface:
![sss.png](attachment:sss.png)
We can see that we need multiple parameters:
- The patient volume that we cant to skull strip
- An output volume to store the volume containing only the brain
- A label map to store the brain mask
- The atlases are optional according to the documentation
So, how do we actually run this module ? And how do we pass these parameters?
Well, all modules can be found in Python from the `slicer.modules` package. Then you have to run this small helper script to learn how the parameters are called:

In [ ]:
import slicer
def print_parameters(module):
    n = module.cliModuleLogic().CreateNode()
    for groupIndex in range(n.GetNumberOfParameterGroups()):
        print(f'Group: {n.GetParameterGroupLabel(groupIndex)}')
        for parameterIndex in range(n.GetNumberOfParametersInGroup(groupIndex)):
          print('  {0} [{1}]: {2}'.format(n.GetParameterName(groupIndex, parameterIndex),
            n.GetParameterTag(groupIndex, parameterIndex),n.GetParameterLabel(groupIndex, parameterIndex)))

# You can use the autocompletion to help you find the module that you're looking for
print_parameters(slicer.modules.swissskullstripper)

Tada! We get the list of parameters for this module and how they are called. Now we only have to build  dictionary containing our parameters and we're good to go:
**Note:** During the computation, 3D Slicer will appear as being unresponsive, but all is actually fine.

In [ ]:
# Build the parameters
output_volume = scene.AddNewNodeByClass("vtkMRMLScalarVolumeNode")
output_mask = scene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")

parameters = {
    "patientVolume": volume,  # Volume from the sample data
    "patientOutputVolume": output_volume,
    "patientMaskLabel": output_mask,
}

# Run the module synchronously
slicer.cli.runSync(slicer.modules.swissskullstripper, None, parameters)

As easy as that! The skull stripping usually takes a minute. When the computation is over, a brain mask should appear in 3D Slicer. Now you know how to completely automatically run a module within 3D Slicer! You could also use the segment editor's different effects to segment a volume. However, this is left to you the reader as an exercise.

**Note:** you can run the module asynchronously, but be careful to wait until the result are written to the volumes before trying to save them to your hard drive. If you need more information about running modules from Python, you can find them [here](https://slicer.readthedocs.io/en/latest/developer_guide/python_faq.html).

**Note:** you can also access the volumes as `numpy` arrays using the `slicer.util.arrayFromVolume(volume)` function.

## Saving Data

The final step is to save the results that we have computed. Saving nodes is as easy as loading them. As usual, working with DICOM files is slightly more tedious:

In [ ]:
import DICOMScalarVolumePlugin

# Get the ID of the output volume
output_volume_id = shNode.GetItemByDataNode(output_volume)
# Set the study for the output volume
shNode.SetItemParent(output_volume_id, another_study_id)
# Instantiate the exporter
dicom_exporter = DICOMScalarVolumePlugin.DICOMScalarVolumePluginClass()
# Export
exportables = dicom_exporter.examineForExport(output_volume_id)
for exp in exportables:
    # set output folder
    exp.directory = sample_dicom_dir

dicom_exporter.export(exportables)

Saving volumes or label maps to Nifti files is a one liner as usual:

In [ ]:
# Saving the volume
assert slicer.util.saveNode(output_volume, os.path.join(temp_dir, "cropped_sample_data.nii.gz"))
# Saving the brain mask
assert slicer.util.saveNode(output_mask, os.path.join(temp_dir, "brain_mask.nii.gz"))

**Note:** If you want to save a segmentation, you first have to convert it to a label map. The main reason is that label maps don't allow segments to overlap and allow for a one to one match between voxels and labels.

In [ ]:
# Create a new label map
segmentation_as_labelmap = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode")
# Necessary steps as the convertion will look for these display nodes
segmentation.CreateDefaultDisplayNodes()
# Convert the segmentation
slicer.modules.segmentations.logic().ExportVisibleSegmentsToLabelmapNode(segmentation, segmentation_as_labelmap, output_mask)
assert slicer.util.saveNode(segmentation_as_labelmap, os.path.join(temp_dir, "segmentation.nii.gz"))

## Closing Words

I hope that this notebook has given you the tools to start automating tasks with 3D Slicer. About everything that you can see, can be interacted with through Python. If you need ideas or just code samples, you can check out the **extensive** script [repository](https://slicer.readthedocs.io/en/latest/developer_guide/script_repository.html). With that being said, I'll leave you with my favorite script. It's especially useful, if you have to look at a lot of head CT images for a challenge and you don't want to manually set the value range for each volume.

In [ ]:
import MRMLCorePython
nodes = getNodesByClass("vtkMRMLScalarVolumeNode")
for node in nodes:
    # getNodesByClass("vtkMRMLScalarVolumeNode") also returns label maps, which we need to filter out
    if type(node) == MRMLCorePython.vtkMRMLScalarVolumeNode and node.GetDisplayNode():
        node.GetDisplayNode().SetAutoWindowLevel(False)
        node.GetDisplayNode().SetWindowLevelMinMax(0, 100)


In [ ]:
!apt-get update
!apt-get install -y fsl

In [ ]:

# Step 2: Set up FSL environment
import os
os.environ["FSLDIR"] = "/usr/share/fsl/5.0"
os.environ["PATH"] += ":/usr/lib/fsl/5.0"

In [ ]:
# Step 4: Define output paths
output_volume_path = "/content/852176.nii.gz"
output_mask_path = "/content"


In [ ]:
# Step 5: Run BET for skull stripping
!bet {input_file_path} {output_volume_path} -m

In [ ]:
import os
import tempfile
import slicer

# Step 1: Load your clinical data
# Replace 'path_to_your_data.nii.gz' with the path to your clinical data file
clinical_data_path = "/content/852176.nii.gz"
clinical_volume_node = slicer.util.loadVolume(clinical_data_path)

if not clinical_volume_node:
    raise ValueError(f"Failed to load clinical data from {clinical_data_path}")

print(f"Clinical data loaded successfully: {clinical_volume_node.GetName()}")

# Step 2: Create a new patient/study in the subject hierarchy
shNode = slicer.vtkMRMLSubjectHierarchyNode.GetSubjectHierarchyNode(slicer.mrmlScene)
volume_id = shNode.GetItemByDataNode(clinical_volume_node)

# Customize patient and study names
patient_id = shNode.CreateSubjectItem(shNode.GetSceneItemID(), "MyPatient")
study_id = shNode.CreateStudyItem(patient_id, "MyStudy")
shNode.SetItemParent(volume_id, study_id)

# Step 3: Export to NIfTI (optional, if needed)
temp_dir = tempfile.gettempdir()
nifti_output_path = os.path.join(temp_dir, "clinical_data.nii.gz")
assert slicer.util.saveNode(clinical_volume_node, nifti_output_path)
print(f"Clinical data exported to NIfTI: {nifti_output_path}")

# Step 4: Run SwissSkullStripper to strip the skull
scene = slicer.mrmlScene

# Create nodes for the output volume and brain mask
output_volume_node = scene.AddNewNodeByClass("vtkMRMLScalarVolumeNode", "SkullStrippedVolume")
output_mask_node = scene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode", "BrainMask")

# Build parameters for SwissSkullStripper
parameters = {
    "patientVolume": clinical_volume_node,  # Input clinical data
    "patientOutputVolume": output_volume_node,  # Output skull-stripped volume
    "patientMaskLabel": output_mask_node,  # Output brain mask
}

# Run the SwissSkullStripper module synchronously
slicer.cli.runSync(slicer.modules.swissskullstripper, None, parameters)

print("Skull stripping completed.")

# Step 5: Save the skull-stripped volume and brain mask
skull_stripped_path = os.path.join(temp_dir, "skull_stripped.nii.gz")
brain_mask_path = os.path.join(temp_dir, "brain_mask.nii.gz")

assert slicer.util.saveNode(output_volume_node, skull_stripped_path)
assert slicer.util.saveNode(output_mask_node, brain_mask_path)

print(f"Skull-stripped volume saved to: {skull_stripped_path}")
print(f"Brain mask saved to: {brain_mask_path}")

# Step 6: Optional - Convert segmentation to label map
segmentation_as_labelmap = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLLabelMapVolumeNode", "SegmentationAsLabelMap")
segmentation = slicer.mrmlScene.AddNewNodeByClass("vtkMRMLSegmentationNode")
segmentation.SetReferenceImageGeometryParameterFromVolumeNode(output_volume_node)
segmentation.CreateDefaultDisplayNodes()

# Export visible segments to label map
slicer.modules.segmentations.logic().ExportVisibleSegmentsToLabelmapNode(segmentation, segmentation_as_labelmap, output_volume_node)

labelmap_output_path = os.path.join(temp_dir, "segmentation.nii.gz")
assert slicer.util.saveNode(segmentation_as_labelmap, labelmap_output_path)

print(f"Segmentation as label map saved to: {labelmap_output_path}")

# Step 7: Clear the subject hierarchy (optional)
shNode.RemoveAllItems()

In [ ]:
# Step 1: Install FSL
!apt-get update
!apt-get install -y fsl

# Step 2: Set up FSL environment
import os
os.environ["FSLDIR"] = "/usr/share/fsl/5.0"
os.environ["PATH"] += ":/usr/lib/fsl/5.0"

# Step 3: Upload your clinical data
from google.colab import files

# Upload your clinical data (e.g., .nii.gz)
# uploaded = files.upload()
input_file_path ="/content/852176.nii.gz"# Get the uploaded file name

# Step 4: Define output paths
output_volume_path = "/content/sample_data/out_image"
output_mask_path = "/content/sample_data/output_mask"

# Step 5: Run BET for skull stripping
!bet {input_file_path} {output_volume_path} -m

print(f"Skull stripping completed. Output saved to {output_volume_path} and {output_mask_path}")

# Step 6: Download the results
files.download(output_volume_path)
files.download(output_mask_path)